# Routing101 Captioning — 3-mode caption search, RRF, mapped back to keyframes

**Purpose:** search the frame-caption layer (not keyframe images, not ASR transcript) three
ways, fuse the three ranked lists with RRF, then resolve each hit to its keyframe for display.

| Mode | Source | Space | Text encoder |
|---|---|---|---|
| 3.1 `clip_caption` | `AICDataExtracted/clip_caption/*_caption_clip.npy` (built into a **new** FAISS index here) | CLIP 512-d | Multilingual-CLIP (`pipeline/clip_encoder.py`) |
| 3.2 `siglip_caption` | `AICDataExtracted/siglip_caption/*_caption_siglip768.npy` (built into a FAISS index here) | SigLIP2 768-d | `google/siglip2-base-patch16-384` text tower |
| 3.3 `fuzzy` | `AICDataExtracted/captioning/*_captions.csv` bulk-indexed into Elasticsearch | lexical | ES `match` query, `fuzziness: AUTO` |
| 3.4 `rrf` | fuses 3.1 + 3.2 + 3.3 | — | — |

Join key (verified on disk): `captioning.frame_id == clip_caption_frames.frame_id ==
siglip_caption_frames.frame_id == map-keyframes.n` — captions are already **per-frame**
(one caption per keyframe), unlike ASR's per-segment granularity, so every leg maps to its
keyframe with a direct `n` lookup — no nearest-timestamp mapping needed anywhere in this
notebook.


## 1. Install / imports
Run once. `pip install --break-system-packages faiss-cpu numpy pandas torch transformers sentencepiece pillow elasticsearch` -- drop the flag if you're in a normal venv/conda env.

In [ ]:
# !pip install --break-system-packages faiss-cpu numpy pandas torch transformers sentencepiece pillow elasticsearch

import sys
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
import torch
from IPython.display import display

print("faiss:", faiss.__version__ if hasattr(faiss, "__version__") else "unknown")
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

# Reuse pipeline/*.py's text encoders (no fusion/store logic in them) --
# same sys.path trick routing101.py / routing101_asr.ipynb use so their
# bare `import config` resolves.
REPO_ROOT = Path.cwd()
PIPELINE_DIR = REPO_ROOT / "pipeline"
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import config as pconfig  # noqa: E402  (pipeline/config.py)
import clip_encoder        # noqa: E402  (pipeline/clip_encoder.py) -- Multilingual-CLIP text tower


## 2. CONFIG — edit this cell

Set `QUERY` and `TOP_K`. Paths point at data outside the repo, same convention as
`pipeline/config.py`. FAISS indices built here are cached on disk under
`index/routing101_caption/` (built once, reused after) -- delete that folder to force a rebuild.

In [ ]:
TOP_K = 100
SHOW_TOP_N = 15

# --- source data ---
CAPTIONING_DIR = Path("D:/University/Summ26/AICDataExtracted/captioning")
CLIP_CAPTION_DIR = Path("D:/University/Summ26/AICDataExtracted/clip_caption")
SIGLIP_CAPTION_DIR = Path("D:/University/Summ26/AICDataExtracted/siglip_caption")
MAP_KEYFRAMES_DIR = Path("D:/University/Summ26/AICData/map-keyframes")
THUMBNAIL_ROOT = Path("D:/University/Summ26/AICData/keyframes")

# --- on-disk FAISS cache for this notebook ---
INDEX_DIR = REPO_ROOT / "index" / "routing101_caption"
INDEX_DIR.mkdir(parents=True, exist_ok=True)
CLIP_CAPTION_FAISS = INDEX_DIR / "clip_caption_flat_ip.index"
CLIP_CAPTION_META = INDEX_DIR / "meta_clip_caption.csv"
SIGLIP_CAPTION_FAISS = INDEX_DIR / "siglip_caption_flat_ip.index"
SIGLIP_CAPTION_META = INDEX_DIR / "meta_siglip_caption.csv"

# --- SigLIP2 text tower (query side only) ---
SIGLIP2_MODEL_ID = "google/siglip2-base-patch16-384"

# --- Elasticsearch (simple local instance, no auth/SSL) ---
# docker run -d --name es -p 9200:9200 -e "discovery.type=single-node" \
#     -e "xpack.security.enabled=false" docker.elastic.co/elasticsearch/elasticsearch:8.15.0
ES_HOST = "http://localhost:9200"
ES_INDEX = "caption_frames"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 3. Shared helpers
`video_id_from_filename` strips a known suffix, `l2_normalize` for cosine-via-inner-product, and
a small per-video `map-keyframes` cache used by the keyframe-lookup step. No nearest-time
mapping helper here (unlike ASR) — captions are already frame-level, so `frame_id ==
map-keyframes.n` directly.

In [ ]:
def video_id_from_filename(npy_path: Path, suffix: str) -> str:
    return npy_path.stem[: -len(suffix)] if npy_path.stem.endswith(suffix) else npy_path.stem


def l2_normalize(mat: np.ndarray) -> np.ndarray:
    mat = mat.astype("float32", copy=True)
    faiss.normalize_L2(mat)
    return mat


_map_keyframes_cache: dict = {}


def load_map_keyframes(video_id: str):
    if video_id not in _map_keyframes_cache:
        path = MAP_KEYFRAMES_DIR / f"{video_id}.csv"
        _map_keyframes_cache[video_id] = pd.read_csv(path) if path.exists() else None
    return _map_keyframes_cache[video_id]


def keyframe_by_n(video_id: str, n: int):
    """Direct n-indexed lookup -- every caption leg carries frame_id == map-keyframes.n."""
    mk = load_map_keyframes(video_id)
    if mk is None or pd.isna(n):
        return None, None
    hit = mk.loc[mk["n"] == int(n)]
    if hit.empty:
        return None, None
    row = hit.iloc[0]
    return float(row["pts_time"]), int(row["frame_idx"])


def thumbnail_path(video_id: str, n) -> str:
    if n is None or pd.isna(n):
        return ""
    return str(THUMBNAIL_ROOT / video_id / f"{int(n):03d}.jpg")


## 4. Step 1 — embed `clip_caption` into a new FAISS index

`clip_caption/*_caption_clip.npy` vectors are precomputed CLIP-space embeddings of frame
captions, one `.npy` (+ `_frames.csv` sidecar of `row_index, frame_id, text`) per video,
**not unit-normalized on disk**. Cached to `index/routing101_caption/clip_caption_flat_ip.index`
+ `meta_clip_caption.csv` so this only runs once.

In [ ]:
def _build_clip_caption_index():
    npy_paths = sorted(CLIP_CAPTION_DIR.glob("*_caption_clip.npy"))
    print(f"[clip_caption] building index over {len(npy_paths)} videos")

    index = faiss.IndexFlatIP(512)
    rows = []
    gid = 0
    for npy_path in npy_paths:
        video_id = video_id_from_filename(npy_path, "_caption_clip")
        frames_path = CLIP_CAPTION_DIR / f"{video_id}_caption_siglip768_frames.csv"
        if not frames_path.exists():
            print(f"  skip {video_id}: missing frames csv")
            continue
        vecs = l2_normalize(np.load(npy_path))
        frames = pd.read_csv(frames_path)
        if len(frames) != vecs.shape[0]:
            print(f"  skip {video_id}: row mismatch ({len(frames)} frames vs {vecs.shape[0]} vectors)")
            continue
        index.add(vecs)
        for _, r in frames.iterrows():
            rows.append((gid, video_id, int(r["frame_id"]), r["text"]))
            gid += 1

    faiss.write_index(index, str(CLIP_CAPTION_FAISS))
    pd.DataFrame(rows, columns=["global_id", "video_id", "frame_id", "text"]).to_csv(CLIP_CAPTION_META, index=False)
    print(f"[clip_caption] done: {gid} frames -> {CLIP_CAPTION_FAISS}")


if not (CLIP_CAPTION_FAISS.exists() and CLIP_CAPTION_META.exists()):
    _build_clip_caption_index()

clip_caption_index = faiss.read_index(str(CLIP_CAPTION_FAISS))
clip_caption_meta = pd.read_csv(CLIP_CAPTION_META)
print("clip_caption index:", clip_caption_index.ntotal, "vectors")


## 5. Mode 3.1 — CLIP caption search
Query text encoded with Multilingual-CLIP (`pipeline/clip_encoder.py`), searched against the
index built in Step 1.

In [ ]:
def search_clip_caption(query: str, k: int = TOP_K) -> pd.DataFrame:
    qvec = l2_normalize(clip_encoder.encode_text([query]))
    n = min(k, clip_caption_index.ntotal)
    scores, ids = clip_caption_index.search(qvec, n)
    rows = []
    for rank, (gid, score) in enumerate(zip(ids[0], scores[0]), start=1):
        if gid == -1:
            continue
        row = clip_caption_meta.iloc[int(gid)]
        rows.append({"rank": rank, "score": float(score), "video_id": row["video_id"],
                      "frame_id": int(row["frame_id"]), "text": row["text"]})
    return pd.DataFrame(rows)


## 6. Mode 3.2 setup — embed `siglip_caption` into a FAISS index

`siglip_caption/*_caption_siglip768.npy` is SigLIP-space, one row per frame caption -- so
`frame_id` in the `_frames.csv` sidecar is already a direct `map-keyframes.n` pointer, no
nearest-timestamp lookup needed for this leg. Cached the same way as Step 1.

In [ ]:
def _build_siglip_caption_index():
    npy_paths = sorted(SIGLIP_CAPTION_DIR.glob("*_caption_siglip768.npy"))
    print(f"[siglip_caption] building index over {len(npy_paths)} videos")

    index = faiss.IndexFlatIP(768)
    rows = []
    gid = 0
    for npy_path in npy_paths:
        video_id = video_id_from_filename(npy_path, "_caption_siglip768")
        frames_path = SIGLIP_CAPTION_DIR / f"{video_id}_caption_siglip768_frames.csv"
        if not frames_path.exists():
            print(f"  skip {video_id}: missing frames csv")
            continue
        vecs = l2_normalize(np.load(npy_path))
        frames = pd.read_csv(frames_path)
        if len(frames) != vecs.shape[0]:
            print(f"  skip {video_id}: row mismatch ({len(frames)} frames vs {vecs.shape[0]} vectors)")
            continue
        index.add(vecs)
        for _, r in frames.iterrows():
            rows.append((gid, video_id, int(r["frame_id"]), r["text"]))
            gid += 1

    faiss.write_index(index, str(SIGLIP_CAPTION_FAISS))
    pd.DataFrame(rows, columns=["global_id", "video_id", "frame_id", "text"]).to_csv(SIGLIP_CAPTION_META, index=False)
    print(f"[siglip_caption] done: {gid} rows -> {SIGLIP_CAPTION_FAISS}")


if not (SIGLIP_CAPTION_FAISS.exists() and SIGLIP_CAPTION_META.exists()):
    _build_siglip_caption_index()

siglip_caption_index = faiss.read_index(str(SIGLIP_CAPTION_FAISS))
siglip_caption_meta = pd.read_csv(SIGLIP_CAPTION_META)
print("siglip_caption index:", siglip_caption_index.ntotal, "vectors")


In [ ]:
_siglip2_state = {}  # lazy singleton: {"model", "processor"}


def _get_siglip2():
    if not _siglip2_state:
        from transformers import AutoModel, AutoProcessor
        model = AutoModel.from_pretrained(SIGLIP2_MODEL_ID).to(DEVICE).eval()
        processor = AutoProcessor.from_pretrained(SIGLIP2_MODEL_ID)
        _siglip2_state.update(model=model, processor=processor)
    return _siglip2_state["model"], _siglip2_state["processor"]


def encode_text_siglip2(texts: list) -> np.ndarray:
    model, processor = _get_siglip2()
    inputs = processor(text=texts, padding="max_length", truncation=True, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.get_text_features(**inputs)
    feats = out.pooler_output if hasattr(out, "pooler_output") else out
    return feats.float().cpu().numpy().astype("float32")


def search_siglip_caption(query: str, k: int = TOP_K) -> pd.DataFrame:
    qvec = l2_normalize(encode_text_siglip2([query]))
    n = min(k, siglip_caption_index.ntotal)
    scores, ids = siglip_caption_index.search(qvec, n)
    rows = []
    for rank, (gid, score) in enumerate(zip(ids[0], scores[0]), start=1):
        if gid == -1:
            continue
        row = siglip_caption_meta.iloc[int(gid)]
        rows.append({"rank": rank, "score": float(score), "video_id": row["video_id"],
                      "frame_id": int(row["frame_id"]), "text": row["text"]})
    return pd.DataFrame(rows)


## 7. Mode 3.3 — Elasticsearch fuzzy search

Simplest possible setup: one flat index over every `captioning/*_captions.csv` row, a single
`match` query with `fuzziness: "AUTO"`. Requires a local ES reachable at `ES_HOST` (see the
docker one-liner in the CONFIG cell). Bulk-index uses an explicit `_id` per doc so re-running
this cell is idempotent (no duplicate docs).

In [ ]:
from elasticsearch import Elasticsearch, helpers

es = Elasticsearch(ES_HOST)


def build_fuzzy_index():
    if not es.indices.exists(index=ES_INDEX):
        es.indices.create(index=ES_INDEX, mappings={"properties": {
            "video_id": {"type": "keyword"},
            "frame_id": {"type": "integer"},
            "text": {"type": "text"},
        }})

    def _docs():
        for csv_path in sorted(CAPTIONING_DIR.glob("*_captions.csv")):
            df = pd.read_csv(csv_path)
            for _, r in df.iterrows():
                yield {
                    "_index": ES_INDEX,
                    "_id": f"{r['video_id']}_{int(r['frame_id'])}",
                    "_source": {
                        "video_id": r["video_id"],
                        "frame_id": int(r["frame_id"]),
                        "text": r["caption_text"],
                    },
                }

    n_ok, errors = helpers.bulk(es, _docs(), stats_only=False, raise_on_error=False)
    print(f"[fuzzy] indexed {n_ok} docs, {len(errors)} errors")


def search_fuzzy(query: str, k: int = TOP_K) -> pd.DataFrame:
    """Returns an empty DataFrame (with a warning) instead of raising when ES
    isn't reachable/indexed -- so a missing local ES only drops this one leg
    rather than breaking the run cell / RRF fusion."""
    try:
        resp = es.search(index=ES_INDEX, size=k, query={
            "match": {"text": {"query": query, "fuzziness": "AUTO"}}
        })
    except Exception as e:
        print(f"[fuzzy] search failed ({e}); returning empty results for this leg.")
        return pd.DataFrame(columns=["rank", "score", "video_id", "frame_id", "text"])

    rows = []
    for rank, hit in enumerate(resp["hits"]["hits"], start=1):
        src = hit["_source"]
        rows.append({"rank": rank, "score": float(hit["_score"]), "video_id": src["video_id"],
                      "frame_id": src["frame_id"], "text": src["text"]})
    return pd.DataFrame(rows)


# Run once to (re-)populate the ES index; safe to re-run (idempotent _id).
try:
    build_fuzzy_index()
except Exception as e:
    print(f"[fuzzy] could not reach Elasticsearch at {ES_HOST}: {e}\n"
          f"        start it first (see docker one-liner in the CONFIG cell) -- "
          f"the other two modes work fine without it.")


## 8. Mode 3.4 — Reciprocal Rank Fusion

Adapted from `routing101_asr.ipynb`'s `rrf_fuse` -- unweighted `1/(k+rank)` per leg, keyed by
`(video_id, frame_id)` (frame-level, since every leg here already shares that granularity —
unlike ASR's segment-level key).

In [ ]:
RRF_K = 60


def rrf_fuse(named_dfs: dict, k: int = RRF_K, top_n: int = TOP_K) -> pd.DataFrame:
    '''named_dfs: {leg_name: DataFrame[rank, video_id, frame_id, text]}'''
    scores: dict = {}
    extra: dict = {}
    for leg_name, df in named_dfs.items():
        if df is None or df.empty:
            continue
        for _, row in df.iterrows():
            key = (row["video_id"], int(row["frame_id"]))
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + row["rank"])
            extra.setdefault(key, {"text": row.get("text")})

    rows = [{"video_id": vid, "frame_id": fid, "rrf_score": s, **extra[(vid, fid)]}
            for (vid, fid), s in scores.items()]
    out = pd.DataFrame(rows).sort_values("rrf_score", ascending=False).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out.head(top_n)


## 9. Map top-k results back to the keyframe

Every leg already carries `frame_id == map-keyframes.n` -- direct lookup, no nearest-timestamp
mapping needed anywhere here.

In [ ]:
def attach_keyframe(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    pts_times, frame_idxs, thumbs = [], [], []
    for _, row in df.iterrows():
        pts_time, frame_idx = keyframe_by_n(row["video_id"], row["frame_id"])
        pts_times.append(pts_time)
        frame_idxs.append(frame_idx)
        thumbs.append(thumbnail_path(row["video_id"], row["frame_id"]))
    out = df.copy()
    out["mapped_pts_time"] = pts_times
    out["mapped_frame_idx"] = frame_idxs
    out["thumbnail_path"] = thumbs
    return out


## 10. Run all 3 modes + RRF, mapped to keyframes

In [ ]:
QUERY = "triển lãm nghệ thuật"

results_clip_caption = attach_keyframe(search_clip_caption(QUERY, k=TOP_K))
results_siglip_caption = attach_keyframe(search_siglip_caption(QUERY, k=TOP_K))
results_fuzzy = attach_keyframe(search_fuzzy(QUERY, k=TOP_K))
results_rrf = attach_keyframe(rrf_fuse({
    "clip_caption": results_clip_caption,
    "siglip_caption": results_siglip_caption,
    "fuzzy": results_fuzzy,
}, top_n=TOP_K))

cols_leg = ["rank", "score", "video_id", "frame_id", "text", "mapped_pts_time"]
cols_rrf = ["rank", "rrf_score", "video_id", "frame_id", "text", "mapped_pts_time"]


def _show(name: str, df: pd.DataFrame, cols: list):
    print(f"--- {name} ---")
    if df is None or df.empty:
        print("(no results)")
    else:
        display(df[cols].head(SHOW_TOP_N))


print(f"QUERY = {QUERY!r}\n")
_show("3.1 clip_caption", results_clip_caption, cols_leg)
_show("3.2 siglip_caption", results_siglip_caption, cols_leg)
_show("3.3 fuzzy (elasticsearch)", results_fuzzy, cols_leg)
_show("3.4 rrf (fused)", results_rrf, cols_rrf)
